# rHEALPix polyline linetrace

Step-by-step visualization of polyline → rHEALPix cells, matching vgrid [`polyline2rhealpix`](https://github.com/opengeoshub/vgrid/blob/main/vgrid/conversion/vector2dggs/vector2rhealpix.py) / [`linetrace`](https://github.com/opengeoshub/vgrid/blob/main/vgrid/dggs/rhealpixdggs/rhp_wrappers.py). Same animation style as [`02_a5_linetrace.ipynb`](02_a5_linetrace.ipynb).

`polyline2rhealpix` calls `linetrace` on each line part, then merges with `seen_ids`.

Input: [`multipolyline.geojson`](https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/multipolyline.geojson) — **5 features**, **6 line parts** (disjoint branches + one closed loop). All rows/parts are loaded and drawn in distinct colors. **rHEALPix resolution** is shown on every frame.

## Install necessary packages

In [ ]:
%pip install vgrid geopandas matplotlib imageio pillow
# optional for MP4:
%pip install imageio-ffmpeg

In [1]:
"""Step-by-step polyline2rhealpix / linetrace animation (multipolyline.geojson)."""
from pathlib import Path

import geopandas as gpd
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
from shapely.geometry import MultiLineString

from vgrid.conversion.dggs2geo.rhealpix2geo import rhealpix2geo
from vgrid.conversion.vector2dggs.vector2rhealpix import polyline2rhealpix
from vgrid.dggs.rhealpixdggs.dggs import RHEALPixDGGS
from vgrid.dggs.rhealpixdggs.rhp_wrappers import linetrace

rhealpix_dggs = RHEALPixDGGS()

URL = "https://raw.githubusercontent.com/opengeoshub/vopendata/main/shape/multipolyline.geojson"
RESOLUTION = 11
PLANE = False
OUT_GIF = "polyline2rhealpix.gif"
OUT_MP4 = "polyline2rhealpix.mp4"
FRAME_EVERY_N_PATH = 5  # 1 = frame per cell; use 5+ for long multipolyline runs
DPI = 120
FIX_ANTIMERIDIAN = None
PART_COLORS = ["#1f4e79", "#c55a11", "#2e7d32", "#b71c1c", "#6a1b9a", "#4e342e"]


def cell_patches(cell_polys, facecolor, edgecolor, alpha=0.55, lw=0.4):
    patches = []
    for poly in cell_polys:
        if poly is None or poly.is_empty:
            continue
        patches.append(MplPolygon(list(poly.exterior.coords), closed=True))
    return PatchCollection(
        patches, facecolor=facecolor, edgecolor=edgecolor, alpha=alpha, linewidths=lw
    )


def polylines_from_feature(feature):
    """Split LineString / MultiLineString the same way as polyline2rhealpix."""
    if feature.geom_type == "LineString":
        return [feature]
    if feature.geom_type == "MultiLineString":
        return list(feature.geoms)
    return []


def polylines_from_gdf(gdf):
    """Flatten every LineString / MultiLineString part across all GeoJSON features."""
    parts = []
    for geom in gdf.geometry:
        if geom is None or geom.is_empty:
            continue
        parts.extend(polylines_from_feature(geom))
    return parts


def feature_from_gdf(gdf):
    """Build one MultiLineString covering all disjoint branches and loops."""
    parts = polylines_from_gdf(gdf)
    if not parts:
        raise ValueError("No line geometries found in input GeoJSON")
    if len(parts) == 1:
        return parts[0]
    return MultiLineString(parts)


def part_color(part_index):
    return PART_COLORS[part_index % len(PART_COLORS)]


def all_waypoints(parts):
    return [(lon, lat) for line in parts for lon, lat in line.coords]


def render_frame(
    parts,
    path_polys,
    title,
    path,
    resolution,
    current_poly=None,
    endpoint_polys=None,
    waypoints=None,
    active_part=None,
):
    fig, ax = plt.subplots(figsize=(8, 8))
    all_bounds = MultiLineString(parts).bounds
    minx, miny, maxx, maxy = all_bounds
    pad = max(maxx - minx, maxy - miny) * 0.08 or 0.01
    ax.set_xlim(minx - pad, maxx + pad)
    ax.set_ylim(miny - pad, maxy + pad)

    for i, line in enumerate(parts):
        color = part_color(i)
        lw = 3.5 if active_part == i else 2.0
        alpha = 1.0 if active_part is None or active_part == i else 0.45
        gpd.GeoSeries([line]).plot(
            ax=ax,
            facecolor="none",
            edgecolor=color,
            lw=lw,
            alpha=alpha,
        )
    if waypoints:
        lons, lats = zip(*waypoints)
        ax.scatter(lons, lats, c="#d62728", s=30, zorder=5)

    visited = list(path_polys) if path_polys else []
    if current_poly is not None:
        visited = [
            p
            for p in visited
            if p is not current_poly and not p.equals(current_poly)
        ]
    if visited:
        ax.add_collection(cell_patches(visited, "#2ca02c", "#1a5f1a", alpha=0.45))

    if endpoint_polys:
        ax.add_collection(
            cell_patches(endpoint_polys, "#d62728", "#8b0000", alpha=0.7)
        )

    if current_poly is not None:
        ax.add_collection(
            cell_patches([current_poly], "#ffcc00", "#cc8800", alpha=0.9, lw=2.5)
        )

    ax.plot([], [], color="#ffcc00", lw=4, label="current cell")
    ax.plot([], [], color="#2ca02c", lw=4, label="path so far")
    ax.plot([], [], color="#d62728", lw=4, label="path endpoints")
    ax.legend(loc="upper right", fontsize=8)
    ax.text(
        0.02,
        0.98,
        f"rHEALPix resolution: {resolution}",
        transform=ax.transAxes,
        fontsize=9,
        va="top",
        ha="left",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.9),
        zorder=6,
    )
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)
    fig.subplots_adjust(left=0.08, right=0.92, top=0.92, bottom=0.08)
    fig.savefig(path, dpi=DPI, facecolor="white")
    plt.close(fig)


def polyline2rhealpix_with_frames(parts, feature, resolution, frame_dir, fix_antimeridian=None):
    """Mirror polyline2rhealpix / linetrace with frame capture for all parts."""
    frame_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    idx = 0

    if not parts:
        return frames, []

    path_polys = []
    ordered_cell_ids = []
    seen_ids = set()

    def snap(
        title,
        current=None,
        endpoints=None,
        show_waypoints=False,
        active_part=None,
        part_waypoints=None,
    ):
        nonlocal idx
        p = frame_dir / f"frame_{idx:04d}.png"
        render_frame(
            parts,
            path_polys,
            title,
            p,
            resolution,
            current_poly=current,
            endpoint_polys=endpoints,
            waypoints=part_waypoints if show_waypoints else None,
            active_part=active_part,
        )
        frames.append(p)
        idx += 1

    n_parts = len(parts)
    n_verts = sum(len(list(line.coords)) for line in parts)
    n_closed = sum(1 for line in parts if line.coords[0] == line.coords[-1])
    snap(
        f"1. Input res {resolution} ({n_parts} parts, {n_verts} vertices, "
        f"{n_closed} closed loop{'s' if n_closed != 1 else ''})"
    )
    snap("2. All parts (distinct colors)", show_waypoints=False)

    for part_i, polyline in enumerate(parts, start=1):
        if polyline.is_empty or polyline.length == 0:
            continue

        is_closed = polyline.coords[0] == polyline.coords[-1]
        part_waypoints = [(lon, lat) for lon, lat in polyline.coords]
        traced = linetrace(polyline, resolution, plane=PLANE, dggs=rhealpix_dggs) or []
        new_ids = []
        for cell_id in traced:
            if cell_id in seen_ids:
                continue
            seen_ids.add(cell_id)
            ordered_cell_ids.append(cell_id)
            new_ids.append(cell_id)

        loop_note = " [closed loop]" if is_closed else ""
        snap(
            f"3. Part {part_i}/{n_parts}{loop_note}: linetrace → "
            f"{len(traced)} cells ({len(new_ids)} new)",
            show_waypoints=True,
            active_part=part_i - 1,
            part_waypoints=part_waypoints,
        )

        for step, cell_id in enumerate(new_ids, start=1):
            cell_poly = rhealpix2geo(cell_id, fix_antimeridian=fix_antimeridian)
            if cell_poly is None or cell_poly.is_empty:
                continue
            path_polys.append(cell_poly)
            if step % FRAME_EVERY_N_PATH == 0 or step == len(new_ids):
                snap(
                    f"4. Part {part_i} step {step}/{len(new_ids)}: {cell_id} "
                    f"({len(ordered_cell_ids)} total)",
                    current=cell_poly,
                    active_part=part_i - 1,
                )

    snap(f"5. Merged linetrace → {len(ordered_cell_ids)} cells")
    snap(f"6. Complete path ({len(ordered_cell_ids)} cells)")

    rows = polyline2rhealpix(feature, resolution, fix_antimeridian=fix_antimeridian)
    from_polyline2rhealpix = [row["rhealpix"] for row in rows]
    if set(from_polyline2rhealpix) != set(ordered_cell_ids):
        print(
            "Warning: cell set differs from polyline2rhealpix:",
            len(ordered_cell_ids),
            "vs",
            len(from_polyline2rhealpix),
        )
    elif from_polyline2rhealpix != ordered_cell_ids:
        print(
            "Note: same cells as polyline2rhealpix, different order "
            "(expected with linetrace vs older BFS vgrid)"
        )

    return frames, ordered_cell_ids


def main():
    gdf = gpd.read_file(URL)
    parts = polylines_from_gdf(gdf)
    feature = feature_from_gdf(gdf)
    print(f"Loaded {len(gdf)} feature(s), {len(parts)} line part(s)")

    frame_dir = Path("_polyline2rhealpix_frames")
    frames, cell_ids = polyline2rhealpix_with_frames(
        parts,
        feature,
        RESOLUTION,
        frame_dir,
        fix_antimeridian=FIX_ANTIMERIDIAN,
    )
    imageio.mimsave(OUT_GIF, [imageio.imread(f) for f in frames], duration=0.9)
    print(f"Wrote {OUT_GIF} ({len(frames)} frames, {len(cell_ids)} final cells)")
    try:
        writer = imageio.get_writer(OUT_MP4, fps=1.2)
        for f in frames:
            writer.append_data(imageio.imread(f))
        writer.close()
        print(f"Wrote {OUT_MP4}")
    except Exception as e:
        print(f"MP4 skipped ({e}). GIF is enough.")


if __name__ == "__main__":
    main()

Loaded 5 feature(s), 6 line part(s)
Note: same cells as polyline2rhealpix, different order (expected with linetrace vs older BFS vgrid)
Wrote polyline2rhealpix.gif (55 frames, 212 final cells)
Wrote polyline2rhealpix.mp4
